# rank-world-size-args — worked example 3: Ring all-reduce: each rank sends right, receives left

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `rank-world-size-args`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

In a ring all-reduce, ranks are arranged in a logical ring. Each rank sends to its right neighbor (`(rank + 1) % world_size`) and receives from its left neighbor (`(rank - 1) % world_size`). This communication pattern is more bandwidth-efficient than the naive all-to-all approach. The signature still uses `(tensor, rank, world_size)` — `rank` and `world_size` determine who each process's ring neighbors are.

## Worked solution

**Step 1 — Compute neighbors.** `right = (rank + 1) % world_size` and `left = (rank - 1) % world_size`. The modular arithmetic handles the wrap-around: rank 0's left neighbor is `world_size - 1`.

**Step 2 — Send to right.** Return `('send', right)` as one action.

**Step 3 — Receive from left.** Return `('recv', left)` as another action.

**Step 4 — Verify the ring structure.** For a 4-rank ring: rank 0 sends to 1 and receives from 3, rank 3 sends to 0 and receives from 2. This confirms the modular wrap-around works.

In [ ]:
import torch as t

def ring_allreduce_step(tensor, rank: int, world_size: int) -> list:
    """One ring step: send right, receive left."""
    right = (rank + 1) % world_size
    left  = (rank - 1) % world_size
    return [('send', right), ('recv', left)]

# Verify ring structure for world_size=4
world_size = 4
for rank in range(world_size):
    actions = ring_allreduce_step(None, rank, world_size)
    print(f'rank={rank}: {actions}')

# Check wrap-around
rank0_actions = ring_allreduce_step(None, 0, 4)
print(f'\nRank 0 sends to:   {rank0_actions[0][1]}')  # 1
print(f'Rank 0 recvs from: {rank0_actions[1][1]}')  # 3 (wrap-around)

rankN_actions = ring_allreduce_step(None, 3, 4)
print(f'Rank 3 sends to:   {rankN_actions[0][1]}')  # 0 (wrap-around)
print(f'Rank 3 recvs from: {rankN_actions[1][1]}')  # 2